<a href="https://colab.research.google.com/github/Francisco1125/r-course/blob/master/Algoritmos_Gen%C3%A9ticos_H%C3%ADper_Parametrizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dependencias

In [ ]:
import numpy as np
import pandas as pd
from itertools import chain
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

## Población Inicial

In [ ]:
def crear_individuo(num_genes=20):
    return "".join([str(int(x>0.5)) for x in np.random.uniform(size=num_genes)])

num_genes = 10
tam_pob = 100
pob = [crear_individuo(num_genes=num_genes) for _ in range(tam_pob)]

## Selección

In [ ]:
def objetivo(x):
    return sum([int(g) for g in x])

def seleccion(objetivo,pob,k = 3):
    pob = pd.DataFrame({'x':pob}).sample(k)
    pob['aptitud'] = pob['x'].map(objetivo)
    pob.sort_values(by='aptitud',inplace=True)
    return pob.tail(1)['x'].values[0]

aptos = [seleccion(objetivo,pob) for _ in range(tam_pob)]

In [ ]:
aptos[:2]

['0000111111', '1000101111']

## Reproducción

In [ ]:
def combinacion(padre1,padre2,proba):
    #Los hijos son por defecto copias de los padres
    padre1 = [g for g in padre1]
    padre2 = [g for g in padre2]
    hijos = [padre1,padre2]
    if np.random.uniform()<proba:
        #Seleccionar un punto de combinación que no esté en los extremos
        i = np.random.choice(range(1,len(padre1)-1))
        hijos = padre1[:i]+padre2[i:],padre2[:i]+padre1[i:]
    return ["".join(h) for h in hijos]

## Mutación

In [ ]:
def mutacion(individuo,proba=0.5):
    return "".join([str(1-int(g)) if r<proba else g  for g,r in zip(individuo,
                                                                    np.random.uniform(size=len(individuo)))])

In [ ]:
hijos = [combinacion(*aptos[i:i+2],0.8) for i in range(0,100,2)]
hijos = list(map(lambda x: mutacion(x),chain(*hijos)))

## Algoritmo genético

In [ ]:
def alg_genetico(objetivo,
                 n_genes=20,
                 n_generaciones=100,
                 tam_pob=100,
                 proba_comb=0.9,
                 proba_mut=1/20):
    pob = [crear_individuo(num_genes=n_genes) for _ in range(tam_pob)]
    for gen in range(n_generaciones):
        aptos = [seleccion(objetivo,pob) for _ in range(tam_pob)]
        hijos = [combinacion(*aptos[i:i+2],proba_comb) for i in range(0,tam_pob,2)]
        hijos = list(map(lambda x: mutacion(x,proba=proba_mut),chain(*hijos)))
        pob = hijos[:]
    pob = pd.DataFrame({'x':pob})
    pob['aptitud'] = pob['x'].map(objetivo)
    pob.sort_values(by='aptitud',inplace=True)
    return pob.tail(1).values.tolist()[0]

In [ ]:
alg_genetico(objetivo,n_generaciones=1)

['10111110111011110110', 15]

## Datos

In [ ]:
data = load_breast_cancer()
X,y = data['data'],data['target']
Xt,Xv,yt,yv = train_test_split(X,y,train_size=0.7)

In [ ]:
Xt.shape,Xv.shape

((398, 30), (171, 30))

## Híper parametrización

In [ ]:
mod = DecisionTreeClassifier()

In [ ]:
espacio_de_busqueda = [[0.01,0.1],[0.01,0.1]]
n_genes = 32
tam_pob = 100
p_mutacion = 1/(n_genes*len(espacio_de_busqueda))


def Z(min_impurity_decrease,min_samples_leaf):
    mod = DecisionTreeClassifier(min_impurity_decrease=min_impurity_decrease,min_samples_leaf=min_samples_leaf)
    mod.fit(Xt,yt)
    return roc_auc_score(y_score=mod.predict_proba(Xv)[:,1],y_true=yv)

def decode(limites,n_genes,x):
    max_ = 2**(n_genes//2)
    x = [int(x[i*n_genes//2:n_genes//2*(i+1)],2) for i in range(len(limites))]
    x = [limites[i][0]+(v/max_)*(limites[i][1]-limites[i][0]) for i,v in enumerate(x)]
    return tuple(x)

def seleccion(objetivo,pob,k = 3,decodificar=False):
    if decodificar:
        decodificados = list(map(lambda x:decode(espacio_de_busqueda,n_genes,x),pob))
        pob = pd.DataFrame({'x':pob,'d':decodificados}).sample(k)
        pob['aptitud'] = pob['d'].map(lambda x:objetivo(*x))
    else:
        pob = pd.DataFrame({'x':pob}).sample(k)
        pob['aptitud'] = pob['x'].map(lambda x:objetivo(x))

    pob.sort_values(by='aptitud',inplace=True)
    return pob.tail(1)['x'].values[0]

pob = [crear_individuo(num_genes=n_genes) for _ in range(tam_pob)]

def alg_genetico(objetivo,
                 n_genes=20,
                 n_generaciones=5,
                 tam_pob=50,
                 proba_comb=0.9,
                 proba_mut=1/20,
                 tam_muestra = 3,
                 decodificar=True):
    pob = [crear_individuo(num_genes=n_genes) for _ in range(tam_pob)]

    for gen in range(n_generaciones):
        print(f'generación:{gen}')
        aptos = [seleccion(objetivo,pob,k=tam_muestra,decodificar=decodificar) for _ in range(tam_pob)]
        hijos = [combinacion(*aptos[i:i+2],proba_comb) for i in range(0,tam_pob,2)]
        hijos = list(map(lambda x: mutacion(x,proba=proba_mut),chain(*hijos)))
        pob = hijos[:]
    if decodificar:
        decodificados = list(map(lambda x:decode(espacio_de_busqueda,n_genes,x),pob))
        pob = pd.DataFrame({'x':pob,'d':decodificados})
        pob['aptitud'] = pob['d'].map(lambda x:objetivo(*x))
    else:
        pob = pd.DataFrame({'x':pob})
        pob['aptitud'] = pob['x'].map(objetivo)
    pob.sort_values(by='aptitud',inplace=True)
    return pob.tail(1).values.tolist()[0]


print(alg_genetico(Z,
                   n_genes=n_genes,
                   tam_pob=tam_pob,
                   proba_mut=p_mutacion,
                   n_generaciones=10,
                   decodificar=True))

generación:0
generación:1
generación:2
generación:3
generación:4
generación:5
generación:6
generación:7
generación:8
generación:9
['00000000000010100001000110101111', (0.01001373291015625, 0.016216888427734374), 0.9796307598413182]


In [ ]:
grid = GridSearchCV(param_grid=dict(min_impurity_decrease=np.arange(0.01,0.1,0.01),
                                     min_samples_leaf=np.arange(0.01,0.1,0.01)),
                    estimator=DecisionTreeClassifier(),
                    cv=3,
                    scoring='roc_auc',
                    verbose=True,
                    n_jobs=-1,)

In [ ]:
grid.fit(Xt,yt)

Fitting 3 folds for each of 81 candidates, totalling 243 fits


GridSearchCV(cv=3, estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'min_impurity_decrease': array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]),
                         'min_samples_leaf': array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09])},
             scoring='roc_auc', verbose=True)

In [ ]:
grid.best_params_,grid.best_score_,grid.best_estimator_

({'min_impurity_decrease': 0.01, 'min_samples_leaf': 0.06999999999999999},
 0.953469579525693,
 DecisionTreeClassifier(min_impurity_decrease=0.01,
                        min_samples_leaf=0.06999999999999999))

In [ ]:
mod = grid.best_estimator_

In [ ]:
roc_auc_score(y_score=mod.predict_proba(Xv)[:,1],y_true=yv)

0.9717729630759842

In [ ]:
from tpot import TPOTClassifier

/home/jose/entornos/ins/lib/python3.8/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")


In [ ]:
tpot = TPOTClassifier(generations=10,
                      cv=3,
                      n_jobs=-1,
                      population_size=100,
                      verbosity=2,
                      scoring='roc_auc')
tpot.fit(Xt, yt)
print(tpot.score(Xv, yv))

Optimization Progress:   0%|          | 0/1100 [00:00<?, ?pipeline/s]

/home/jose/entornos/ins/lib/python3.8/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")
/home/jose/entornos/ins/lib/python3.8/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")
/home/jose/entornos/ins/lib/python3.8/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")
/home/jose/entornos/ins/lib/python3.8/site-packages/tpot/builtins/__init__.py:36: UserWarning: Warning: optional dependency `torch` is n


Generation 1 - Current best internal CV score: 0.9896339684585606

Generation 2 - Current best internal CV score: 0.991729391205984

Generation 3 - Current best internal CV score: 0.9917859352606152

Generation 4 - Current best internal CV score: 0.9917859352606152

Generation 5 - Current best internal CV score: 0.9918573533537134

Generation 6 - Current best internal CV score: 0.9920160097640304

Generation 7 - Current best internal CV score: 0.9926683480007248

Generation 8 - Current best internal CV score: 0.9926683480007248

Generation 9 - Current best internal CV score: 0.9932228884883116

Generation 10 - Current best internal CV score: 0.9932228884883116

Best pipeline: GradientBoostingClassifier(PolynomialFeatures(MLPClassifier(BernoulliNB(input_matrix, alpha=0.001, fit_prior=False), alpha=0.001, learning_rate_init=0.01), degree=2, include_bias=False, interaction_only=False), learning_rate=0.1, max_depth=8, max_features=0.05, min_samples_leaf=4, min_samples_split=15, n_estimato

In [ ]:
tpot.fitted_pipeline_

Pipeline(steps=[('stackingestimator-1',
                 StackingEstimator(estimator=BernoulliNB(alpha=0.001,
                                                         fit_prior=False))),
                ('stackingestimator-2',
                 StackingEstimator(estimator=MLPClassifier(alpha=0.001,
                                                           learning_rate_init=0.01))),
                ('polynomialfeatures', PolynomialFeatures(include_bias=False)),
                ('gradientboostingclassifier',
                 GradientBoostingClassifier(max_depth=8, max_features=0.05,
                                            min_samples_leaf=4,
                                            min_samples_split=15,
                                            subsample=0.8))])

In [ ]:
tpot.export('tpot_exp.py')